# Uygurca ses modelini eğit (ince ayar)

Bu notebook klavyedeki **mevcut** modeli (model/uyghur_asr_fp32.onnx) PyTorch'a çevirir ve yeni verilerle eğitmeye devam eder.
Sonunda uygulamaya doğrudan koyulabilecek iki dosya çıkar:
- `uyghur_asr_int8.onnx`: yeni model
- `uyghur_char_lm.txt`: daha büyük metinle kurulmuş dil modeli

**Önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.

Ayrıntılar için repodaki `TRAINING.md` dosyasına bak.

## 1) Kurulum ve Google Drive
Eğitim ilerlemesi Drive'a kaydedilir. Oturum kapanırsa kaldığın yerden devam edersin.

In [ ]:
!nvidia-smi -L
!pip -q install librosa soundfile onnx onnxruntime py7zr
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/uyghur_asr'      # checkpoint'ler ve sonuçlar burada
import os; os.makedirs(WORK, exist_ok=True)

## 2) Repoyu indir
Repo özelse `REPO_URL` içine token ekle: `https://<TOKEN>@github.com/forgotten1432/uyghur-keyboard`

In [ ]:
REPO_URL = 'https://github.com/forgotten1432/uyghur-keyboard'
!apt-get -qq install -y git-lfs > /dev/null   # Colab no longer ships git-lfs
!git lfs install
!rm -rf /content/repo && GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 $REPO_URL /content/repo
!cd /content/repo && git lfs pull --include="model/*"
!ls -la /content/repo/model
import sys; sys.path.insert(0, '/content/repo/tools/train'); sys.path.insert(0, '/content/repo/tools')

## 3) Veri

### a) THUYG-20 (~20 saat, otomatik iner)
Mevcut model zaten bununla eğitildi. Unutmasın diye karışıma eklenir.

In [ ]:
import py7zr, glob
if not os.path.exists('/content/thuyg/thuyg20_data'):
    !mkdir -p /content/thuyg && wget -q -O /content/thuyg/d.7z https://github.com/gheyret/uyghur-asr-ctc/releases/download/data/thuyg20_data.7z
    !wget -q -O /content/thuyg/train.csv https://raw.githubusercontent.com/gheyret/uyghur-asr-ctc/master/thuyg20_train.csv
    !wget -q -O /content/thuyg/test.csv https://raw.githubusercontent.com/gheyret/uyghur-asr-ctc/master/thuyg20_test.csv
    with py7zr.SevenZipFile('/content/thuyg/d.7z') as z: z.extractall('/content/thuyg')
    os.remove('/content/thuyg/d.7z')

def thuyg_items(csv):
    out = []
    for line in open(csv, encoding='utf-8-sig'):
        p, t = line.rstrip('\n').split('\t')
        out.append(('/content/thuyg/thuyg20_data/' + p.replace('\\', '/').split('/')[-1], t))
    return out
thuyg_train = thuyg_items('/content/thuyg/train.csv')
thuyg_test = thuyg_items('/content/thuyg/test.csv')
print(len(thuyg_train), len(thuyg_test))

### b) Common Voice Uygurca (yüzlerce saat, en önemli kısım)
1. https://datacollective.mozillafoundation.org adresinde hesap aç ve **Credentials** sayfasından bir API anahtarı oluştur.
2. **Common Voice Scripted Speech – Uyghur** veri setinin sayfasında koşulları kabul et. *Download → API / Python Access → Python Library* sekmesinin altındaki **Dataset ID**'yi aşağıya yaz.
3. Colab'da soldaki 🔑 **Secrets** bölümüne `MDC_API_KEY` adıyla anahtarını ekle ve *Notebook access*'i aç.

Veri (~10 GB) doğrudan Colab'ın diskine iner; Drive'a yüklemen gerekmez. Arapça yazılı cümleler otomatik olarak modelin kullandığı Latin yazıya (ULY) çevrilir.

In [ ]:
!pip -q install datacollective==0.7.1
from google.colab import userdata
os.environ['MDC_API_KEY'] = userdata.get('MDC_API_KEY')
os.environ['MDC_DOWNLOAD_PATH'] = '/content/mdc'   # Colab'ın kendi diski
from datacollective import download_dataset
DATASET_ID = 'cmu5x7x6w00duo107puutvcvu'          # Common Voice Scripted Speech 27.0 - Uyghur
cv_root = download_dataset(DATASET_ID, extract=True)
CV_DIR = os.path.dirname(glob.glob(f'{cv_root}/**/train.tsv', recursive=True)[0])
print(CV_DIR)

In [ ]:
import csv, random
from uyghur_train import uey_to_uly, normalize_uly
def cv_items(split):
    out = []
    with open(f'{CV_DIR}/{split}.tsv', encoding='utf-8') as f:
        for row in csv.DictReader(f, delimiter='\t', quoting=csv.QUOTE_NONE):
            t = normalize_uly(uey_to_uly(row['sentence']))
            if t: out.append((f"{CV_DIR}/clips/{row['path']}", t))
    return out
cv_train, cv_dev, cv_test = cv_items('train'), cv_items('dev'), cv_items('test')
print(len(cv_train), len(cv_dev), len(cv_test))
print(cv_train[0])

### c) (İsteğe bağlı ama çok etkili) Kendi sesin
Telefonla 100–300 cümle okuyup kaydet. Her cümleyi ayrı bir dosyaya kaydet (m4a/mp3/wav olur).
Bir `kendi_sesim.tsv` dosyası yaz. Her satır `dosya_adı<TAB>cümle` biçiminde olsun; cümle Arapça ya da Latin yazıyla olabilir.
Hepsini Drive'da `uyghur_asr/kendi_sesim/` klasörüne koy. Model en çok senin sesine ve telefonunun mikrofonuna alışmış olur.

In [ ]:
own = []
own_dir = WORK + '/kendi_sesim'
if os.path.exists(own_dir + '/kendi_sesim.tsv'):
    for line in open(own_dir + '/kendi_sesim.tsv', encoding='utf-8'):
        if '\t' not in line: continue
        f, t = line.rstrip('\n').split('\t', 1)
        t = normalize_uly(uey_to_uly(t)) if any('\u0600' <= c <= '\u06ff' for c in t) else normalize_uly(t)
        own.append((own_dir + '/' + f, t))
print('kendi kayıtların:', len(own))

## 4) Eğitim karışımı
Ücretsiz Colab oturumları birkaç saatle sınırlı. O yüzden Common Voice'tan bir alt küme alınır; her oturumda farklı bir parça kullanılır.
Kendi kayıtların 5 kez tekrarlanır, böylece etkileri artar.

In [ ]:
CV_PER_SESSION = 40000      # bir oturumda kullanılacak CV kaydı (~50 saat)
SESSION = 0                 # her yeni oturumda 1 artır → farklı CV parçası
random.Random(0).shuffle(cv_train)
part = cv_train[SESSION * CV_PER_SESSION:(SESSION + 1) * CV_PER_SESSION]
train_items = part + random.Random(1).sample(thuyg_train, 4000) + own * 5
random.Random(2).shuffle(train_items)
dev_items = random.Random(3).sample(cv_dev, 400) + random.Random(4).sample(thuyg_test, 200)
print('train', len(train_items), 'dev', len(dev_items))

## 5) Modeli yükle ve eğit
Oturum düşerse 1, 2, 3 ve 4. hücreleri tekrar çalıştır. Bu hücre Drive'daki son checkpoint'ten devam eder.

In [ ]:
import torch
from model_from_onnx import UyghurASR
from uyghur_train import SpeechSet, collate, train, evaluate
device = torch.device('cuda')
model = UyghurASR.from_onnx('/content/repo/model/uyghur_asr_fp32.onnx')
CKPT = WORK + '/finetuned.pt'
if os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location='cpu', weights_only=True)); print('checkpoint yüklendi')
model.to(device)
mk = lambda items, tr, bs: torch.utils.data.DataLoader(SpeechSet(items, train=tr), batch_size=bs, shuffle=tr,
                                                       collate_fn=collate, num_workers=2, pin_memory=True)
train_loader, dev_loader = mk(train_items, True, 16), mk(dev_items, False, 16)
best = train(model, train_loader, dev_loader, device, epochs=3, lr=1e-5, out_path=CKPT, max_minutes=180)
print('en iyi dev CER', best)

## 6) ONNX'e aktar ve int8'e küçült

In [ ]:
model.cpu().eval()
model.export_onnx(WORK + '/uyghur_asr_fp32_new.onnx')
calib = WORK + '/calib'; os.makedirs(calib, exist_ok=True)
import librosa, soundfile as sf
for i, (p, _) in enumerate(dev_items[:40]):
    y, _ = librosa.load(p, sr=22050); sf.write(f'{calib}/{i}.wav', y, 22050)
!cd /content/repo/tools && python quantize_int8.py "$WORK/uyghur_asr_fp32_new.onnx" "$WORK/uyghur_asr_int8.onnx" "$WORK/calib"

## 7) Dil modelini büyük metinle yeniden kur
Common Voice'un tüm cümleleri ile THUYG metinleri birleştirilir. Bu, sesli veriden bağımsız olarak hata oranını düşürür.

In [ ]:
sents = set()
for f in ['validated.tsv', 'train.tsv', 'dev.tsv', 'other.tsv', 'validated_sentences.tsv']:
    path = f'{CV_DIR}/{f}'
    if not os.path.exists(path): continue
    with open(path, encoding='utf-8') as fh:
        for row in csv.DictReader(fh, delimiter='\t', quoting=csv.QUOTE_NONE):
            t = normalize_uly(uey_to_uly(row.get('sentence') or ''))
            if t: sents.add(t)
test_sents = {t for _, t in cv_test} | {normalize_uly(t) for _, t in thuyg_test}
with open('/content/lm_text.tsv', 'w', encoding='utf-8') as o:
    for t in sents | {normalize_uly(t) for _, t in thuyg_train}:
        if t not in test_sents: o.write(t + '\n')     # test cümleleri LM'e girmesin (dürüst ölçüm)
print(len(sents), 'cümle')
!cd /content/repo/tools && python export_char_lm.py "$WORK/uyghur_char_lm.txt" 6 8 /content/lm_text.tsv
!ls -la $WORK

## 8) Ölç: eski model ile yeni modeli karşılaştır
Bir önceki modele göre WER düşmediyse yeni modeli uygulamaya koyma.

In [ ]:
with open('/content/test.tsv', 'w', encoding='utf-8') as o:
    for p, t in random.Random(9).sample(cv_test, 300): o.write(f'{p}\t{t}\n')
!cd /content/repo/tools && echo ESKİ && python evaluate.py /content/repo/app/src/main/assets/uyghur_asr_int8.onnx /content/test.tsv --lm /content/repo/app/src/main/assets/uyghur_char_lm.txt 2>/dev/null || true
!cd /content/repo/tools && echo YENİ && python evaluate.py "$WORK/uyghur_asr_int8.onnx" /content/test.tsv --lm "$WORK/uyghur_char_lm.txt"

## 9) Uygulamaya koy
**Seçenek A (kolay):** GitHub'da Settings → Developer settings → Personal access tokens bölümünden, bu repoya *Contents: write* izni olan bir token oluştur ve aşağıya yapıştır. Hücre dosyaları yeni bir branch'e push eder. Sonra Actions'tan APK'yı derle.

**Seçenek B:** İki dosyayı Drive'dan indir. Mac'te `app/src/main/assets/` içine kopyalayıp `git add`, `commit` ve `push` yap (repo LFS kullanıyor).

In [ ]:
TOKEN = ''   # github token
if TOKEN:
    !cd /content/repo && git lfs pull --include="app/src/main/assets/*" && git checkout -b model-update
    !cp "$WORK/uyghur_asr_int8.onnx" "$WORK/uyghur_char_lm.txt" /content/repo/app/src/main/assets/
    !cd /content/repo && git -c user.name=colab -c user.email=colab@example.com commit -am "Fine-tuned model"
    !cd /content/repo && git push https://$TOKEN@github.com/forgotten1432/uyghur-keyboard model-update